In [1]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 01. Dataset Acquisition, Physical Inventory & Multi-Source Audit\n",
    "**Project:** SIH26038 — Explainable AI for Diabetic Retinopathy Screening in Rural India\n",
    "**Objective:** Inventory raw datasets, compute SHA-256 checksums, screen perceptual duplicates, audit label distributions, and confirm dataset boundaries without model training."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import hashlib\n",
    "from pathlib import Path\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "from PIL import Image\n",
    "import yaml\n",
    "\n",
    "# Load base project configuration\n",
    "with open('../configs/base.yaml', 'r') as f:\n",
    "    base_cfg = yaml.safe_load(f)\n",
    "\n",
    "data_dir = Path('../data')\n",
    "print(f\"Project: {base_cfg['project']['name']} ({base_cfg['project']['sih_problem_id']})\")\n",
    "print(f\"Data Directory Root: {data_dir.resolve()}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 1. Ingest Manifests and Verify File Integrity"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def audit_image_file(image_path: Path):\n",
    "    \"\"\"Compute SHA-256 and read image physical dimensions without modifications.\"\"\"\n",
    "    hasher = hashlib.sha256()\n",
    "    with open(image_path, 'rb') as f:\n",
    "        buf = f.read(65536)\n",
    "        while len(buf) > 0:\n",
    "            hasher.update(buf)\n",
    "            buf = f.read(65536)\n",
    "    sha256_hash = hasher.hexdigest()\n",
    "    \n",
    "    with Image.open(image_path) as img:\n",
    "        width, height = img.size\n",
    "        channels = len(img.getbands())\n",
    "        img_format = img.format\n",
    "        \n",
    "    return {\n",
    "        'filename': image_path.name,\n",
    "        'sha256': sha256_hash,\n",
    "        'size_bytes': image_path.stat().st_size,\n",
    "        'width': width,\n",
    "        'height': height,\n",
    "        'channels': channels,\n",
    "        'format': img_format\n",
    "    }\n",
    "\n",
    "print(\"Audit utility functions initialized.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 2. Multi-Class ICDR and Screening Decision Distribution Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load audited dataset summary artifact\n",
    "summary_df = pd.read_csv(data_dir / 'dataset_audit' / 'dataset_summary.csv')\n",
    "display(summary_df)\n",
    "\n",
    "# Confirm MVP Referable DR Definition: ICDR >= 2\n",
    "referable_threshold = base_cfg['data']['referable_threshold_grade']\n",
    "print(f\"Approved MVP Referable Threshold: ICDR Grade >= {referable_threshold}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 3. Messidor-2 External Quarantine Confirmation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Verify Messidor-2 is isolated from training pipelines\n",
    "messidor_external = summary_df.loc[summary_df['dataset'] == 'messidor2', 'quarantined_external'].values[0]\n",
    "assert messidor_external == True, \"CRITICAL: Messidor-2 must be quarantined as external validation!\"\n",
    "print(\"CONFIRMED: Messidor-2 is quarantined for Phase 12 zero-tuning evaluation.\")"
   ]
  }
 ],
 "metadata": {
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}

NameError: name 'null' is not defined